# Small Models, Same Rules - clean run (2 judges, checkpointed)

**Run the cells top to bottom.** This version is crash-proof and self-correcting:

- Each model is scored by **two judges**: the HarmBench classifier (judge of record) and a keyword baseline. (Llama-Guard-3-1B was dropped - it flagged everything "unsafe" regardless of the answer, so it carried no signal.)
- Every model **checkpoints to Google Drive every 25 prompts**. If Colab sleeps or crashes, just re-run that one cell - it skips finished prompts and continues. Sleeping costs you nothing.
- Every model cell **fixes its own working directory** (`%cd /content/repo`), so the "can't open 05_experiment.py" error can't happen.

Use a **freshly rotated** Hugging Face token (the old one was exposed).

## Step 0 - Pick a GPU
Runtime -> Change runtime type -> **L4 GPU**. The next cell confirms it.

In [ ]:
!nvidia-smi

## Step 1 - Clone the code
Clones `main`, which has the checkpoint/resume feature.

In [ ]:
REPO_URL = "https://github.com/shubmittal/Jailbreak-Robustness-Small-LLMs.git"
!rm -rf /content/repo
!git clone $REPO_URL /content/repo
%cd /content/repo
!git checkout main
!git log -1 --oneline

## Step 2 - Install dependencies (a couple of minutes)

In [ ]:
%cd /content/repo
!pip install -q -r requirements.txt

## Step 3 - Hugging Face login

First accept the licenses (same account as your token): `meta-llama/Llama-3.2-3B-Instruct`, `google/gemma-2-2b-it`, the `walledai/HarmBench` **dataset**, and `cais/HarmBench-Llama-2-13b-cls`. Use a **Read** token (a fine-grained token will fail on the gated dataset). Then run and paste a freshly rotated token.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 4 - Mount Google Drive
All results and checkpoints are written here, so nothing is lost on a reset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/jailbreak_results/run

## Step 5 - Smoke test (~2 min) - Gemma canary
Confirms Gemma's license works before the long run. Throwaway output.

In [ ]:
%cd /content/repo
!python 05_experiment.py --models google/gemma-2-2b-it --model-revisions 299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8 --judges keyword --n 2 --max_new_tokens 16 --no_plot --output_dir ./results/smoke

## Step 6 - Run the grid, one model per cell

Each cell scores 800 completions (200 HarmBench + 200 XSTest x 2 conditions) with both judges and streams results + a checkpoint to Drive. **If one crashes or sleeps, just re-run that same cell** - it resumes from the last checkpoint (you'll see a `[resume] loaded ...` line). Roughly 30-60 min each on an L4.

(Llama-3.2-3B already finished on an earlier run and is on Drive, so you can skip its cell if you like - re-running it just regenerates the same data.)

In [ ]:
%cd /content/repo
!python 05_experiment.py --backend transformers --models meta-llama/Llama-3.2-3B-Instruct --model-revisions 0cb88a4f764b7a12671c53f0838cd831a0843b95 --judges keyword,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --checkpoint-every 25 --output_dir /content/drive/MyDrive/jailbreak_results/run/llama
print('Llama-3.2-3B cell finished - check the output ABOVE for the aggregate table, not this line')

In [ ]:
%cd /content/repo
!python 05_experiment.py --backend transformers --models microsoft/Phi-3-mini-4k-instruct --model-revisions f39ac1d28e925b323eae81227eaba4464caced4e --judges keyword,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --checkpoint-every 25 --output_dir /content/drive/MyDrive/jailbreak_results/run/phi3
print('Phi-3-mini cell finished - check the output ABOVE for the aggregate table, not this line')

In [ ]:
%cd /content/repo
!python 05_experiment.py --backend transformers --models Qwen/Qwen2.5-3B-Instruct --model-revisions aa8e72537993ba99e69dfaafa59ed015b17504d1 --judges keyword,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --checkpoint-every 25 --output_dir /content/drive/MyDrive/jailbreak_results/run/qwen
print('Qwen2.5-3B cell finished - check the output ABOVE for the aggregate table, not this line')

In [ ]:
%cd /content/repo
!python 05_experiment.py --backend transformers --models google/gemma-2-2b-it --model-revisions 299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8 --judges keyword,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --checkpoint-every 25 --output_dir /content/drive/MyDrive/jailbreak_results/run/gemma
print('Gemma-2-2B cell finished - check the output ABOVE for the aggregate table, not this line')

## Step 7 - Merge the four models

In [ ]:
import pandas as pd, os
root = '/content/drive/MyDrive/jailbreak_results/run'
parts = []
for name in ['llama', 'phi3', 'qwen', 'gemma']:
    p = f'{root}/{name}/results.csv'
    if os.path.exists(p):
        d = pd.read_csv(p)
        d = d[d['judge'] != 'llamaguard']   # drop the dead judge if any old rows have it
        parts.append(d); print(name, len(d), 'rows')
    else:
        print('MISSING:', p)
combined = pd.concat(parts, ignore_index=True)
os.makedirs(f'{root}/combined', exist_ok=True)
combined.to_csv(f'{root}/combined/results.csv', index=False)
print('TOTAL rows:', len(combined), '(expect 6400 = 4 models x 800 x 2 judges)')
print(combined['model'].value_counts())
print(combined['judge'].value_counts())

## Step 8 - Analysis (tables, confidence intervals, figures)
Outputs land in a `results/` subfolder inside `combined/`.

In [ ]:
%cd /content/repo
!python 06_analysis.py --results-csv /content/drive/MyDrive/jailbreak_results/run/combined/results.csv --out-dir /content/drive/MyDrive/jailbreak_results/run/combined --bootstrap 1000 --ci 0.95

## Step 9 - Verify outputs

In [ ]:
!ls -la /content/drive/MyDrive/jailbreak_results/run/combined/results
import json
try:
    s = json.load(open('/content/drive/MyDrive/jailbreak_results/run/combined/results/summary.json'))
    print(json.dumps(s, indent=2)[:2000])
except Exception as e:
    print('summary.json not found yet:', e)

## Troubleshooting & notes

- **`can't open 05_experiment.py`** - the kernel's directory reset. Each cell already starts with `%cd /content/repo`, so just re-run the cell. If it persists, re-run Step 1.
- **A cell crashes or Colab sleeps** - re-run that same cell. It resumes from the last checkpoint on Drive (look for `[resume] loaded ...`).
- **Runtime fully reset** - re-run Steps 1-4, then re-run only the model cells you hadn't finished.
- **Out-of-memory on the 13B judge** - add `--harmbench-cls-size small` to that model's command.
- **Optional keep-awake** (browser console, F12 -> Console; unreliable, the checkpoint is the real safety net):
  ```js
  setInterval(() => document.querySelector('colab-connect-button')?.shadowRoot?.querySelector('#connect')?.click(), 60000);
  ```
- **HF token** - use a freshly rotated **Read** token; the old one was exposed.